# symbol-firewall: is a SYNC-style latent symbol channel an injection firewall, an attack surface, or both?
**Branch:** `feature/symbol-firewall` | **Gate:** pre-registered in README.md (frozen 2026-11-08)
**One-Click Execution:** Runtime -> Run all (L4). Clones, installs, mounts Drive (results + HF cache), runs pytest on the GPU
(aborts if the write hook is invalid on the real receiver), runs the resumable campaign, prints RESULTS.md, unassigns the GPU.
Re-running after a disconnect resumes from Drive (finished stage files are skipped). Budget: <= 15 CU (expected ~4-6).

In [ ]:
# Step 1: Mount Drive; results (incl. resume state) and the HF model cache (~16 GB, downloaded once) live there
import os  # first cell: the Drive prompt appears right away
from google.colab import drive
drive.mount("/content/drive")
os.environ["SF_RESULTS"] = "/content/drive/MyDrive/moonshot_results/symbol-firewall"
os.environ["HF_HOME"] = "/content/hf_cache"
os.makedirs(os.environ["SF_RESULTS"], exist_ok=True)
print("Results dir:", os.environ["SF_RESULTS"], "| existing:", sorted(os.listdir(os.environ["SF_RESULTS"])))

In [ ]:
# Step 2: Credit Guard (auto-unassign on any error or completion) + Colab Secrets
import os, sys, signal, subprocess
from IPython import get_ipython

def auto_teardown(signum=None, frame=None):
    print("\n[Credit Guard] Error, interrupt, or completion! Flushing Drive and unassigning GPU...")
    try:
        from google.colab import drive
        drive.flush_and_unmount()
    except Exception:
        pass
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

def custom_exc_handler(self, etype, value, tb, tb_offset=None):
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)
    auto_teardown()
    return []

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), custom_exc_handler)

try:
    from google.colab import userdata
    for k in ["HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "hf_token"]:
        try:
            tok = userdata.get(k)
            if tok:
                os.environ["HF_TOKEN"] = tok
                os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
                print(f"[Secrets] Loaded Hugging Face token from Colab Secrets ({k}).")
                break
        except Exception:
            pass
except ImportError:
    pass

print("[Ready] Credit guard active. VM will self-terminate on error or completion.")

In [ ]:
# Step 3: GPU check + clone repository
import torch
assert torch.cuda.is_available(), "Switch Colab runtime to GPU (L4 24GB): Runtime -> Change runtime type -> GPU (L4)"
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB | torch {torch.__version__} | CUDA {torch.version.cuda}")

repo_url = "https://github.com/jaimejarauta/decoupled-delta-net.git"
branch = "feature/symbol-firewall"
target_dir = "/content/symbol-firewall"
if not os.path.exists(target_dir):
    subprocess.run(["git", "clone", "-b", branch, repo_url, target_dir], check=True)
else:
    subprocess.run(["git", "-C", target_dir, "fetch", "origin", branch], check=True)
    subprocess.run(["git", "-C", target_dir, "reset", "--hard", f"origin/{branch}"], check=True)
os.chdir(target_dir)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Step 4: Installs
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if os.environ.get("HF_TOKEN"):
    import huggingface_hub
    huggingface_hub.login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print(subprocess.run([sys.executable, "-c", "import sf, transformers; print('transformers', transformers.__version__, '| device', sf.DEV, sf.DTYPE)"],
                     capture_output=True, text=True, check=True).stdout)

In [ ]:
# Step 5: Pre-flight on the GPU (bf16, real receiver Qwen3-4B-Instruct-2507 + tiny model): zero write is bit-identical,
# write only affects positions >= first slot, padded batch == single row, parsers, gate rules, end-to-end smoke.
# ABORTS (and unassigns) if anything fails. run_all.py repeats it and records pytest.json.
_t = subprocess.run([sys.executable, "-m", "pytest", "-v", "-p", "no:cacheprovider", "test_sf.py"], capture_output=True, text=True)  # Colab hides child output
print(_t.stdout[-20000:]); print(_t.stderr[-5000:])
os.makedirs("/content/drive/MyDrive/moonshot_results/symbol-firewall", exist_ok=True); open("/content/drive/MyDrive/moonshot_results/symbol-firewall/pytest_log.txt", "w").write(_t.stdout + _t.stderr)
if _t.returncode:
    raise RuntimeError("pytest failed on GPU: write hook / scoring invalid -> campaign aborted")
print("[Pre-flight] all checks passed on GPU.")

In [ ]:
# Step 6: Full campaign (resumable): senders -> directions/probe/pilot (early stop) -> Side A -> Side B -> gate
p = subprocess.Popen([sys.executable, "-u", "run_all.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if "Loading weights" not in line:
        print(line, end="")
if p.wait():
    raise RuntimeError(f"run_all.py failed (exit {p.returncode}); re-run the notebook to resume from Drive")

In [ ]:
# Step 7: Results
print(open(os.path.join(os.environ["SF_RESULTS"], "RESULTS.md")).read())

In [ ]:
# Step 8: Flush Drive & unassign GPU
auto_teardown()